# Entity Resolution Analysis
Visualizes results from the LSH sweep (Optimization #2) and classifier evaluation.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='Set2')

_cwd = Path.cwd().resolve()
if os.environ.get("RESULTS_DIR"):
    RESULTS_DIR = Path(os.environ["RESULTS_DIR"]).expanduser().resolve()
elif (_cwd / "data").is_dir():
    RESULTS_DIR = (_cwd / "data" / "results").resolve()
else:
    RESULTS_DIR = (_cwd.parent / "data" / "results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. LSH Sweep: Recall vs Speedup

In [ ]:
sweep = pd.read_csv(str(RESULTS_DIR / 'er_lsh_sweep.csv'))
sweep.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recall vs numHashTables (for threshold=0.4)
subset = sweep[sweep['threshold'] == 0.4]
axes[0].plot(subset['num_hash_tables'], subset['recall_approx'], 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('numHashTables', fontsize=12)
axes[0].set_ylabel('Approximate Recall', fontsize=12)
axes[0].set_title('Recall vs numHashTables (threshold=0.4)', fontsize=13)
axes[0].set_ylim(0, 1.05)
axes[0].grid(True)

# Speedup heatmap
pivot = sweep.pivot(index='num_hash_tables', columns='threshold', values='speedup_x')
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Speedup vs Brute-Force (higher=better)', fontsize=13)
axes[1].set_xlabel('Threshold', fontsize=12)
axes[1].set_ylabel('numHashTables', fontsize=12)

plt.suptitle('MinHash LSH Parameter Sweep', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'er_lsh_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Recall–Speedup Trade-off (Pareto curve)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('tab10', len(sweep['num_hash_tables'].unique()))
for i, (nht, group) in enumerate(sweep.groupby('num_hash_tables')):
    ax.scatter(group['speedup_x'], group['recall_approx'],
               label=f'b={nht}', s=80, color=colors[i])
    ax.plot(group.sort_values('speedup_x')['speedup_x'],
            group.sort_values('speedup_x')['recall_approx'],
            '--', alpha=0.5, color=colors[i])

ax.set_xlabel('Speedup over brute-force (×)', fontsize=12)
ax.set_ylabel('Approximate Recall', fontsize=12)
ax.set_title('Recall–Speedup Trade-off (LSH)', fontsize=13)
ax.legend(title='numHashTables')
ax.axhline(y=0.9, color='red', linestyle=':', label='90% recall threshold')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'er_pareto.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Classifier Evaluation

In [ ]:
eval_df = pd.read_csv(str(RESULTS_DIR / 'er_evaluation.csv'))
eval_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(eval_df))
width = 0.25

ax.bar(x - width, eval_df['precision'], width, label='Precision')
ax.bar(x, eval_df['recall'], width, label='Recall')
ax.bar(x + width, eval_df['f1'], width, label='F1')

ax.set_xticks(x)
ax.set_xticklabels(eval_df['entity_type'])
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('ER Classifier: Precision / Recall / F1', fontsize=13)
ax.legend()

for bars in ax.containers:
    ax.bar_label(bars, fmt='%.2f', fontsize=9)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'er_classifier.png'), dpi=150, bbox_inches='tight')
plt.show()